# Outlier Removal – data_campaign_flat.csv

Removes rows flagged by any of the following criteria (evaluated **per pond**, in time order):

| Criterion | Rule |
|-----------|------|
| Time gap | Rows immediately following a gap > 20 min since the previous reading |
| DO low | DO < 0.1 mg/L |
| DO jump | |ΔDO| > 2 mg/L between consecutive readings (~15 min) |
| DO high AM | DO > 10 mg/L before noon |
| pH range | pH < 6 or pH > 10 |
| pH jump | |ΔpH| > 1 between consecutive readings (~15 min) |

In [1]:
import pandas as pd
import numpy as np

INPUT_PATH  = 'data/data_campaign_flat.csv'
OUTPUT_PATH = 'data/data_campaign_flat_clean.csv'

## 1. Load & parse

In [2]:
df = pd.read_csv(INPUT_PATH)

# combine date + time into a single datetime column
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])

# sort within each pond by time
df = df.sort_values(['pond_id', 'datetime']).reset_index(drop=True)

print(f'Loaded {len(df):,} rows across {df["pond_id"].nunique()} ponds')
df.head()

Loaded 72,750 rows across 17 ponds


,pond_id,date,time,do_mg_l,ph,temperature_c,datetime
0,pond_319c1ff7,2025-12-14,02:15:00,6.51,8.75,24.9,2025-12-14 02:15:00
1,pond_319c1ff7,2025-12-14,02:30:00,6.45,8.76,24.9,2025-12-14 02:30:00
2,pond_319c1ff7,2025-12-14,02:45:00,6.40,8.76,24.9,2025-12-14 02:45:00
3,pond_319c1ff7,2025-12-14,03:00:00,6.37,8.76,24.8,2025-12-14 03:00:00
4,pond_319c1ff7,2025-12-14,03:15:00,6.31,8.76,24.8,2025-12-14 03:15:00


## 2. Flag outliers

In [3]:
# ── per-pond diff helpers ────────────────────────────────────────────────────
g = df.groupby('pond_id', sort=False)

time_diff_min = g['datetime'].diff().dt.total_seconds().div(60)  # minutes
do_diff       = g['do_mg_l'].diff().abs()
ph_diff       = g['ph'].diff().abs()

# ── individual flags ────────────────────────────────────────────────────────
flag_time_gap  = time_diff_min > 20
flag_do_low    = df['do_mg_l'] < 0.1
flag_do_jump   = do_diff > 2
flag_do_high_am = (df['do_mg_l'] > 10) & (df['datetime'].dt.hour < 12)
flag_ph_range  = (df['ph'] < 6) | (df['ph'] > 10)
flag_ph_jump   = ph_diff > 1

# ── combined flag ───────────────────────────────────────────────────────────
any_flag = (
    flag_time_gap  |
    flag_do_low    |
    flag_do_jump   |
    flag_do_high_am|
    flag_ph_range  |
    flag_ph_jump
)

# ── summary ─────────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'criterion': [
        'time_gap > 20 min',
        'DO < 0.1 mg/L',
        'DO jump > 2 mg/L',
        'DO > 10 mg/L before noon',
        'pH < 6 or > 10',
        'pH jump > 1',
    ],
    'rows_flagged': [
        flag_time_gap.sum(),
        flag_do_low.sum(),
        flag_do_jump.sum(),
        flag_do_high_am.sum(),
        flag_ph_range.sum(),
        flag_ph_jump.sum(),
    ]
})
summary['pct_of_total'] = (summary['rows_flagged'] / len(df) * 100).round(2)
print(summary.to_string(index=False))
print(f'\nTotal rows flagged (any criterion): {any_flag.sum():,}  ({any_flag.mean()*100:.2f}%)')

               criterion  rows_flagged  pct_of_total
       time_gap > 20 min          1645          2.26
           DO < 0.1 mg/L           272          0.37
        DO jump > 2 mg/L          2082          2.86
DO > 10 mg/L before noon          1193          1.64
          pH < 6 or > 10           541          0.74
             pH jump > 1            50          0.07

Total rows flagged (any criterion): 5,421  (7.45%)


## 3. Per-pond breakdown

In [4]:
pond_summary = df.assign(
    flag_time_gap   = flag_time_gap,
    flag_do_low     = flag_do_low,
    flag_do_jump    = flag_do_jump,
    flag_do_high_am = flag_do_high_am,
    flag_ph_range   = flag_ph_range,
    flag_ph_jump    = flag_ph_jump,
    any_flag        = any_flag,
).groupby('pond_id').agg(
    total_rows       = ('pond_id',        'size'),
    time_gap         = ('flag_time_gap',   'sum'),
    do_low           = ('flag_do_low',     'sum'),
    do_jump          = ('flag_do_jump',    'sum'),
    do_high_am       = ('flag_do_high_am', 'sum'),
    ph_range         = ('flag_ph_range',   'sum'),
    ph_jump          = ('flag_ph_jump',    'sum'),
    total_flagged    = ('any_flag',        'sum'),
)
pond_summary['pct_flagged'] = (pond_summary['total_flagged'] / pond_summary['total_rows'] * 100).round(2)
print(pond_summary.to_string())

               total_rows  time_gap  do_low  do_jump  do_high_am  ph_range  ph_jump  total_flagged  pct_flagged
pond_id                                                                                                        
pond_319c1ff7        4149       158      15      116          13         1        3            288         6.94
pond_35f0d376        4435       118      24       56          15         1        2            210         4.74
pond_44865e41        2338        31       0       40           4         0        2             74         3.17
pond_46bbdb3a        5584       209      49      239          47         3        4            500         8.95
pond_522cd38a        5477        70       0       80         146         0        0            282         5.15
pond_56e8a695        4414       134       1      130          34         1        2            281         6.37
pond_5f07dc7a        2298        98       1       76          72         1        2            236      

## 4. Remove flagged rows & save

In [5]:
df_clean = df[~any_flag].drop(columns='datetime').reset_index(drop=True)

print(f'Original rows : {len(df):,}')
print(f'Removed rows  : {any_flag.sum():,}')
print(f'Clean rows    : {len(df_clean):,}')

df_clean.to_csv(OUTPUT_PATH, index=False)
print(f'\nSaved to {OUTPUT_PATH}')

Original rows : 72,750
Removed rows  : 5,421
Clean rows    : 67,329

Saved to data/data_campaign_flat_clean.csv


## 5. Quick sanity check on clean data

In [6]:
print('DO range :', df_clean['do_mg_l'].min(), '–', df_clean['do_mg_l'].max())
print('pH range :', df_clean['ph'].min(), '–', df_clean['ph'].max())
print('Ponds    :', df_clean['pond_id'].nunique())
df_clean.describe()

DO range : 0.1 – 25.56
pH range : 6.0 – 9.24
Ponds    : 17


,do_mg_l,ph,temperature_c
count,67329.000000,67329.000000,67329.000000
mean,6.887062,8.483097,25.973099
std,4.577713,0.290416,0.990167
min,0.100000,6.000000,0.000000
25%,3.310000,8.320000,25.400000
50%,5.970000,8.510000,26.000000
75%,9.830000,8.690000,26.600000
max,25.560000,9.240000,32.400000
